# 🏭 Conveyor Belt Defect AI - High Precision YOLOv8 Retraining
This notebook trains a high-precision **YOLOv8 Small (`yolov8s`)** or **Medium (`yolov8m`)** model for detecting industrial conveyor belt defects.

### Supported Defect Classes:
1. **Large Hole** (`Sobek/Lubang Besar`)
2. **Small Hole** (`Lubang Kecil`)
3. **Belt Joint** (`Sambungan Belt`)
4. **Large Tear** (`Sobekan Besar`)
5. **Small Tear** (`Sobekan Kecil`)

In [ ]:
# Step 1: Verify Free GPU in Google Colab
# Go to Runtime -> Change runtime type -> Select T4 GPU
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics YOLO
!pip install -q ultralytics

### Step 3: Prepare Your Dataset
Choose **Option A** (Upload zip from computer) or **Option B** (Roboflow export).

In [ ]:
# Option A: Upload dataset.zip directly from your computer
from google.colab import files
import zipfile, os

print("Please upload your dataset.zip file containing images & labels:")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print(f"Extracted {filename} into 'dataset/' successfully!")

In [ ]:
# Option B: Download directly from Roboflow (if using Roboflow)
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# version = project.version(1)
# dataset = version.download("yolov8")

### Step 4: Train with High Precision Hyperparameters
We upgrade from `yolov8n` (Nano - 3.2M params) to `yolov8s` (Small - 11.2M params) with industrial data augmentation (mosaic, lighting variations, contrast scaling) to eliminate missed defects.

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8s or fine-tune from existing best.pt
model = YOLO('yolov8s.pt')

# Train model
results = model.train(
    data='dataset/data.yaml',
    epochs=80,
    batch=16,
    imgsz=640,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    project='runs/train',
    name='conveyor_defect_v2'
)

In [ ]:
# Step 5: Evaluate Precision & Validation Metrics
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 6: Download the newly trained best.pt model to your computer
from google.colab import files
files.download('runs/train/conveyor_defect_v2/weights/best.pt')